<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Regresión lineal**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🚗 Regresión lineal múltiple: millas por galón (auto-mpg)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Estimar el consumo (`mpg`) de un auto a partir de sus características técnicas. |
| 🔍 Comprensión de los datos | Exploramos `auto-mpg.csv`: faltantes, rangos, dispersión por pares y matriz de correlación. |
| 🧹 Preparación de los datos | Eliminamos faltantes, dividimos entrenamiento/prueba y filtramos observaciones fuera de rango. |
| 🤖 Modelado | Ajustamos la regresión múltiple con las variables más correlacionadas (`hp`, `weight`, …). |
| 📊 Evaluación | Interpretamos el resumen del modelo y revisamos residuos y errores de predicción. |
| 🚀 Despliegue y conclusiones | Predecimos el consumo de autos nuevos y resumimos qué porcentaje de la variabilidad explica el modelo. |

> 🎯 **Analogía:** el consumo de combustible de un auto es como una receta: caballos de fuerza, peso y cilindros son ingredientes que suben o bajan las millas por galón.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('auto-mpg.csv', 'dia_1')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Estimar el consumo (`mpg`) de un auto a partir de sus características técnicas.

📚 Lectura recomendada: [3]

## Análisis del Problema
El desempeño de un automóvil se puede medir de diferentes formas. Algunas comunes son la cantidad de caballos de fuerza y el rendimiento del mismo, que se puede resumir en cuantas millas puede recorrer el automóvil por cada galón de combustible que consume. Para los clientes, potenciales compradores de un automóvil, este rendimiento es importante pues puede ayudar a tomar una decisión con respecto a cuál automóvil comprar (si, por ejemplo, el cliente quiere un auto que rinda por muchas millas y pueda economizar en la compra de combustible).

Desde este punto de vista, tanto a clientes como a fabricadores de automóviles, les conviene entender cuál es la relación entre diferentes características del automóvil y su rendimiento, pues el conocer estas relaciones les puede ayudar a inferir cuál va a ser la eficiencia del vehículo a partir de ver los valores de otras características. Para fabricantes, puede ser importante conocer estas relaciones para saber cómo hacer cada modelo más eficiente con respecto al anterior.



# Regresión Lineal Múltiple
Los modelos lineales son una de las herramientas más importantes del análisis cuantitativo. Los utilizamos cuando queremos predecir –o explicar– una variable dependiente a partir de una o más variables independientes. El uso mismo de la terminología de variables dependientes e independientes nos da una pista del abordaje: el ajuste de modelos lineales busca encontrar una función –en el sentido matemático– que nos permite expresar a una variable a partir de otras.

### Autores
   * Efrén Jiménez

###Librerias



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
import plotly.graph_objs as go

---
## 🔍 Fase 2 · Comprensión de los datos

Exploramos `auto-mpg.csv`: faltantes, rangos, dispersión por pares y matriz de correlación.

📚 Lectura recomendada: [4]

## Entendimiento de los Datos

Con el fin de analizar y tratar de estimar las millas por galón de diferentes modelos de automóviles, se trabajó con un conjunto de datos que contiene 398 observaciones y 9 variables:

- mpg (millas por galón): numérica, con un rango de 9 a 46.60.
- cyl (cilindraje): categórica ordinal, con valores posibles de 3, 4, 5, 6 y 8.
- disp (desplazamiento): numérica, con un rango de 68 a 455.
- hp (caballos de fuerza): numérica, con un rango de 46 a 230 y 6 valores faltantes.
- weight (peso): numérica, con un rango de 1613 a 5140.
- acc (aceleración): numérica, con un rango de 8 a 24.80.
- model year (año): categórica, con 13 valores diferentes representando el año del automóvil.
- origin (origen): categórica, 3 valores posibles: 1, 2, 3.
- model name (nombre del modelo): categórica, con 305 posibles valores.

In [ ]:
#cargue el archivo a una variable que se llame autos usando la función read.table
autos = pd.read_csv('sample_data/auto-mpg.csv', header=0, na_values='?')

In [ ]:
corr_matrix = autos.iloc[:,0:7].corr()

print(corr_matrix)

---
## 🧹 Fase 3 · Preparación de los datos

Eliminamos faltantes, dividimos entrenamiento/prueba y filtramos observaciones fuera de rango.

📚 Lectura recomendada: [4]

In [ ]:
autos = autos.iloc[:,0:6]

# Eliminamos las filas con valores faltantes
autos = autos.dropna()

# Cambiamos las variables que corresponden a numéricas
autos['mpg'] = pd.to_numeric(autos['mpg'], errors='coerce')
autos['disp'] = pd.to_numeric(autos['disp'], errors='coerce')
autos['hp'] = pd.to_numeric(autos['hp'], errors='coerce')
autos['weight'] = pd.to_numeric(autos['weight'], errors='coerce')
autos['acc'] = pd.to_numeric(autos['acc'], errors='coerce')

# Dividimos el conjunto de datos en uno de entrenamiento y otro de pruebas
np.random.seed(1376)
mask = np.random.rand(len(autos)) < 0.7
autos_entrenamiento = autos[mask]
autos_prueba = autos[~mask]

print('----------------------------- Entrenamiento -----------------------------')
print(autos_entrenamiento.info())
print('------------------------------- Pruebas ----------------------------------')
print(autos_prueba.info())

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

In [ ]:
autos_prueba = autos_prueba[autos_prueba['hp'] <= 230]

In [ ]:
# Resumen del conjunto de entrenamiento
print(".................................Entrenamiento..................................")
print(autos_entrenamiento.describe())

# Resumen del conjunto de prueba
print(".................................Pruebas........................................")
#autos_prueba = autos_prueba[autos_prueba['acc'] <= 9]
print(autos_prueba.describe())

De acuerdo con los resúmenes anteriores, hay algunas observaciones en el conjunto de datos de prueba cuyo rango de las variables disp y weight se extiende más allá del rango en el conjunto de datos de entrenamiento, así que vamos a eliminar esas observaciones del conjunto de datos de prueba.

In [ ]:
# Eliminar datos fuera de rango del conjunto de prueba
print("Eliminar datos fuera de rango")
autos_prueba = autos_prueba[(autos_prueba['weight'] >= 1649) & (autos_prueba['disp'] >= 70)]

# Resumen del conjunto de entrenamiento
print(".................................Entrenamiento..................................")
print(autos_entrenamiento.describe())

# Resumen del conjunto de prueba
print(".................................Pruebas........................................")
print(autos_prueba.describe())

En total, se eliminaron 2 observaciones.

Para trabajar con regresiones lineales, es importante trabajar solo con variables cuantitativas y estudiar las relaciones que hay entre ellas. Con esto en mente, podemos comenzar nuestra exploración creando gráficos de dispersión para ver cuál es la relación entre nuestra variable de interés (mpg) y el resto de las variables cuantitativas:

---
## 🔍 Fase 2 · Comprensión de los datos — relaciones entre variables

Exploramos `auto-mpg.csv`: faltantes, rangos, dispersión por pares y matriz de correlación.

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(8, 6))

axs[0, 0].scatter(autos_entrenamiento['disp'], autos_entrenamiento['mpg'])
axs[0, 0].set(title='Relación entre mpg y disp', xlabel='disp', ylabel='mpg')

axs[0, 1].scatter(autos_entrenamiento['hp'], autos_entrenamiento['mpg'])
axs[0, 1].set(title='Relación entre mpg y hp', xlabel='hp', ylabel='mpg')

axs[1, 0].scatter(autos_entrenamiento['weight'], autos_entrenamiento['mpg'])
axs[1, 0].set(title='Relación entre mpg y weight', xlabel='weight', ylabel='mpg')

axs[1, 1].scatter(autos_entrenamiento['acc'], autos_entrenamiento['mpg'])
axs[1, 1].set(title='Relación entre mpg y acc', xlabel='acc', ylabel='mpg')

plt.tight_layout()
plt.show()

En los gráficos creados anteriormente, podemos ver como si existe algún tipo de relación, aunque no sea exactamente lineal, entre mpg y las otras cuatro variables cuantitativas. De estas cuatro variables, la que parece tener menor relación es la variable acc con la variable mpg.

También es importante visualizar la relación entre las diferentes variables predictoras, para lo cual podemos crear una matriz de gráficos de dispersión:

In [ ]:
autos_df = pd.DataFrame(autos_entrenamiento, columns=['mpg', 'cyl', 'disp', 'hp', 'weight', 'acc'])

sns.pairplot(autos_df, diag_kind='hist')
plt.suptitle('Relación entre predictores',y=1.05)
plt.show()

![Mapa de calor de correlaciones](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_04.png)
*Figura: así se lee un mapa de calor de correlaciones (los valores cercanos a ±1 indican relación fuerte). [7]*

La información del gráfico anterior podemos complementarla con una matriz de correlación:

In [ ]:
autos_df = pd.DataFrame(autos_entrenamiento, columns=['mpg', 'cyl', 'disp', 'hp', 'weight', 'acc'])
correlations = autos_df.corr()

print(correlations)

In [ ]:
sns.heatmap(correlations, cmap='coolwarm', annot=True)
plt.title('Matriz de correlación')
plt.show()

Como pudimos apreciar en la matriz de gráficos de dispersión, y confirmar con la matriz de correlación, hay una correlación significativa entre los variables cly con (disp, hp, weight) y viceversa. La única variable que no tiene una correlación absoluta mayor a 0.75 es acc. Dado esto, debemos escoger una variable entre hp, disp y weight, para lo cual podemos crear una matriz de correlación entre esas tres variables y mpg para ver cuál tiene una correlación mayor con mpg:

In [ ]:
# Seleccionar las columnas de interés
autos_df = pd.DataFrame(autos_entrenamiento, columns=['mpg', 'cyl', 'disp', 'hp', 'weight'])

# Seleccionar las filas sin valores perdidos en la variable 'hp'
autos_df = autos_df.loc[autos_df['hp'].notna()]

# Calcular la matriz de correlación
correlations = autos_df.corr()

print(correlations)

In [ ]:
sns.heatmap(correlations, cmap='coolwarm', annot=True)
plt.title('Matriz de correlación')
plt.show()

Basándonos en la correlación absoluta, se va a escoger las variables cyl, disp, hp, weight para ser incluida en el modelo.

---
## 🤖 Fase 4 · Modelado

Ajustamos la regresión múltiple con las variables más correlacionadas (`hp`, `weight`, …).

📚 Lectura recomendada: [6] [5]

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

In [ ]:
# Crear la matriz de predictores X y la variable de respuesta y
X = autos_entrenamiento[['hp','weight']].dropna() # Seleccionamos solo las filas sin valores perdidos
y = autos_entrenamiento.loc[X.index]['mpg'] # Seleccionamos la variable 'mpg' correspondiente a las filas seleccionadas

# Ajustar el modelo de regresión lineal
reg_mpg = sm.OLS(y, sm.add_constant(X)).fit()

# Imprimir los resultados
print(reg_mpg.summary())

### Notas importantes

#### Nota:


El modelo generado por la regresión lineal ajustada tiene la siguiente forma:

**mpg = 44.91 - 0.019 * hp - 0.005 * weight**


---



Aquí, mpg es la variable de respuesta que estamos tratando de predecir, mientras que hp y weight son las variables predictoras. El modelo estima los valores de los coeficientes de estas variables, que son -0.019 para hp y -0.005 para weight.


---



El término constante es 44.91, que indica el valor esperado de mpg cuando hp y weight son iguales a cero. En este caso, el valor de 44.91 es poco significativo en términos de interpretación, ya que los valores de hp y weight son significativamente mayores que cero en el conjunto de datos.


---



Por lo tanto, podemos interpretar los coeficientes de la siguiente manera: un aumento de una unidad en hp se asocia con una disminución de 0.019 en mpg, manteniendo todas las demás variables constantes. De manera similar, un aumento de una unidad en weight se asocia con una disminución de 0.005 en mpg, manteniendo todas las demás variables constantes.


---



El R-cuadrado del modelo es 0.703, lo que indica que el modelo explica el 70.3% de la variación en mpg. Esto sugiere que el modelo es relativamente bueno en la predicción de mpg a partir de hp y weight, aunque aún hay cierta variación no explicada en los datos.



---



El valor p para cada coeficiente es menor que 0.05, lo que indica que ambos coeficientes son significativos y diferentes de cero. Por lo tanto, podemos concluir que hp y weight son predictores significativos de mpg en este modelo.

En el resumen del modelo, podemos ver que ambas variables son significativas y que el modelo creado explica alrededor de un 70% de la variación en la variable de respuesta (mpg). Asimismo, podemos ver que el modelo es mejor que un modelo sin variables. Con este modelo, procedemos a hacer las predicciones sobre el conjunto de datos de prueba.

---
## 📊 Fase 5 · Evaluación

Interpretamos el resumen del modelo y revisamos residuos y errores de predicción.

📚 Lectura recomendada: [5] [7]

Se realiza una evaluación de los residuos de la regresión lineal.

### Residuos: ¿qué buscamos?

Un buen modelo deja residuos sin estructura, centrados en 0. Una forma curva indica que falta capturar una relación no lineal.

![Residuos con patrón curvo: el modelo lineal no captura toda la relación](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_09.png)
*Figura: Residuos con patrón curvo: el modelo lineal no captura toda la relación. [7]*

![Residuos repartidos al azar alrededor de 0](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_15.png)
*Figura: Residuos repartidos al azar alrededor de 0. [7]*

In [ ]:
# Calcular residuos estandarizados
residuos = reg_mpg.get_influence().resid_studentized_internal

# Generar gráficos para visualizar residuos estandarizados
fig, ax = plt.subplots(1, 3, figsize=(12,4))
ax[0].hist(residuos)
ax[0].set_title('Histograma')
ax[1].boxplot(residuos)
ax[1].set_title('Diagrama de cajas')
sm.qqplot(residuos, line='s', ax=ax[2])
ax[2].set_title('Gráfico de cuantiles')
plt.show()

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=autos['hp'],
    y=autos['weight'],
    z=autos['mpg'],
    mode='markers',
    marker=dict(
        size=5,
        color=autos['mpg'],
        colorscale='Viridis',
        opacity=0.8
    )
)])

fig.update_layout(
    title='3D Scatterplot',
    scene=dict(
        xaxis_title='hp',
        yaxis_title='weight',
        zaxis_title='mpg'
    ),
    margin=dict(l=0, r=0, b=0, t=0)
)

fig.show()

In [ ]:
# Predecir los valores de mpg para los datos de prueba
X_prueba = autos_prueba[['hp', 'weight']]
X_prueba = sm.add_constant(X_prueba)
autos_prueba['Prediccion'] = reg_mpg.predict(X_prueba)

# Calcular los residuos
autos_prueba['Residuos'] = autos_prueba['mpg'] - autos_prueba['Prediccion']

# Crear un histograma de los residuos
plt.hist(autos_prueba['Residuos'], bins=50)

# Añadir etiquetas y título
plt.xlabel('Residuos')
plt.ylabel('Frecuencia')
plt.title('Distribución de los residuos en prueba')

# Mostrar el gráfico
plt.show()


In [ ]:
plt.scatter(x = autos_prueba['mpg'], y = autos_prueba['mpg'] - autos_prueba['Prediccion'])
plt.title('Distribución de los residuos por mpg')
plt.xlabel('mpg')
plt.ylabel('residuos')
plt.show()

Se realizará el proceso para obtener valores de predicción a partir del modelo de regresión lineal multiple generado.

In [ ]:
print(".................................Mayor..................................")
# Seleccionar las filas donde mpg es mayor o igual que la predicción
autoss = autos_prueba[autos_prueba['mpg'] >= autos_prueba['Prediccion']]
print(autoss)

print(".................................Menor..................................")

# Seleccionar las filas donde mpg es menor o igual que la predicción
autoss2 = autos_prueba[autos_prueba['mpg'] <= autos_prueba['Prediccion']]
print(autoss2)

Un auto con 52 caballos de fuera y 3000 kilos, hace alrededor de 25 millas por galon

In [ ]:
print(reg_mpg.predict([[1,52,3000]]))

---
## 🚀 Fase 6 · Despliegue y conclusiones

Predecimos el consumo de autos nuevos y resumimos qué porcentaje de la variabilidad explica el modelo.

📚 Lectura recomendada: [4]

## Resultados
De acuerdo con la evaluación hecha, el modelo solo predice un 70% de la variabilidad de los datos, por lo cual no se recomienda su uso en ambientes donde se necesite una precision mas alta. Debe mejorar la calidad de los datos debido a que muchos de estos datos no tienen una alta correlacion entre si, que permita subir el rango de predicción de los datos.

---
## 🧪 Refuerzo: multicolinealidad: factor de inflación de la varianza (VIF)

Cuando dos predictoras están muy correlacionadas (por ejemplo, `disp` y `cyl`), los coeficientes se vuelven **inestables**. El **VIF** de una variable mide cuánto se infla la varianza de su coeficiente por esa redundancia: $\text{VIF}_j=\frac{1}{1-R_j^2}$. Regla práctica: VIF > 5 (o 10) indica problema [5].

In [ ]:
import pandas as pd, statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

cols = ['cyl', 'disp', 'hp', 'weight']
Xv = sm.add_constant(autos_entrenamiento[cols].astype(float))
vif = pd.DataFrame({'variable': Xv.columns,
                    'VIF': [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])]})
print(vif[vif.variable != 'const'].round(2).to_string(index=False))

### ✍️ Ejercicios
1. Quite la variable con mayor VIF y vuelva a ajustar el modelo: ¿cambia mucho el R²?
2. ¿Por qué quitar una variable redundante puede *mejorar* la interpretación aunque el R² baje un poco?

---
## ✅ Conclusiones y siguientes pasos

La regresión múltiple permite combinar variables, pero exige vigilar la **multicolinealidad** (variables muy correlacionadas entre sí) [5].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8], [9].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Dua, D., & Graff, C. (2019). *UCI Machine Learning Repository*. University of California, Irvine. https://archive.ics.uci.edu

**[4]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[5]** Faraway, J. J. (2014). *Linear Models with R* (2.ª ed.). CRC Press.

**[6]** Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

**[7]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[8]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[9]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021